# Fault-Tolerant Stream Processing with DLQ

This notebook demonstrates `RobustStreamProcessor` for fault-tolerant stream processing with automatic recovery and dead-letter queue (DLQ) support.

## Features

- **Crash Recovery** - Automatically claim pending messages from crashed consumers
- **Dead-Letter Queue** - Move failed messages to DLQ after max retries
- **Message Replay** - Replay DLQ messages back to main stream
- **Health Monitoring** - Get processor health statistics

In [ ]:
import os
import asyncio
from redis.asyncio import Redis

from redis_openai_agents import RobustStreamProcessor

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
STREAM_NAME = "demo_robust_stream"

print(f"Redis URL: {REDIS_URL}")

## 1. Initialize the Processor

Create a robust stream processor with configurable retry settings.

In [ ]:
processor = RobustStreamProcessor(
    redis_url=REDIS_URL,
    stream_name=STREAM_NAME,
    consumer_group="demo_workers",
    consumer_name="worker_demo",
    dlq_stream=f"{STREAM_NAME}:dlq",  # Dead-letter queue
    max_retries=3,  # Move to DLQ after 3 failed attempts
    claim_timeout_ms=5000,  # Claim messages idle for 5+ seconds
)

await processor.initialize()
print("Processor initialized")
print(f"  Stream: {STREAM_NAME}")
print(f"  DLQ: {STREAM_NAME}:dlq")
print(f"  Max retries: 3")

## 2. Add Test Messages

Add some messages to the stream for processing.

In [ ]:
# Direct Redis client for adding messages
client = Redis.from_url(REDIS_URL, decode_responses=True)

# Add test messages
messages = [
    {"type": "task", "action": "process_order", "order_id": "ORD001"},
    {"type": "task", "action": "send_email", "email": "user@example.com"},
    {"type": "task", "action": "update_inventory", "sku": "PROD123"},
    {"type": "task", "action": "fail_always", "test": "should_go_to_dlq"},  # Will fail
]

print("Adding messages to stream:")
print("=" * 40)
for msg in messages:
    msg_id = await client.xadd(STREAM_NAME, msg)
    print(f"  Added: {msg_id} - {msg['action']}")

await client.aclose()

## 3. Process Messages with Handler

Define a message handler and process the stream.

In [ ]:
# Message handler
async def handle_message(msg: dict) -> bool:
    """Process a message. Return True on success, False to retry."""
    action = msg.get("action", "unknown")
    
    # Simulate processing
    print(f"  Processing: {action}")
    
    # Simulate failure for specific action
    if action == "fail_always":
        print(f"    -> FAILED (will retry)")
        return False
    
    await asyncio.sleep(0.05)  # Simulate work
    print(f"    -> SUCCESS")
    return True

# Process messages
print("Processing messages:")
print("=" * 40)

processed = await processor.process_batch(
    handler=handle_message,
    batch_size=10,
    block_ms=1000,
    max_batches=3,  # Process up to 3 batches then stop
)

print(f"\nSuccessfully processed: {processed} messages")

## 4. Health Statistics

Monitor the processor's health.

In [ ]:
stats = await processor.get_health_stats()

print("Processor Health:")
print("=" * 40)
print(f"  Stream length: {stats['stream_length']}")
print(f"  Pending messages: {stats['pending_messages']}")
print(f"  Consumers: {stats['consumers']}")
print(f"  DLQ length: {stats['dlq_length']}")
print(f"  Last delivered ID: {stats['last_delivered_id']}")

## 5. Claim Pending Messages

Recover messages from crashed/slow consumers.

In [ ]:
# Wait for messages to become "old" enough to claim
print("Waiting for pending messages to become claimable...")
await asyncio.sleep(6)  # Wait for claim_timeout_ms

# Claim pending messages
claimed = await processor.claim_pending_messages()

print(f"\nClaimed {claimed} messages for reprocessing")

# Check health again
stats = await processor.get_health_stats()
print(f"DLQ length after claiming: {stats['dlq_length']}")

## 6. View Dead-Letter Queue

Inspect messages that failed after max retries.

In [ ]:
# Get DLQ messages
dlq_messages = await processor.get_dlq_messages(count=10)

print("Dead-Letter Queue Messages:")
print("=" * 60)

if not dlq_messages:
    print("  (No messages in DLQ yet - may need more retries)")
else:
    for msg in dlq_messages:
        print(f"\n  ID: {msg.get('id')}")
        print(f"  Action: {msg.get('action')}")
        print(f"  Failure reason: {msg.get('failure_reason')}")
        print(f"  Attempts: {msg.get('attempts')}")
        print(f"  Original stream: {msg.get('original_stream')}")

## 7. Replay DLQ Message

After fixing the issue, replay a DLQ message back to the main stream.

In [ ]:
# Get DLQ messages
dlq_messages = await processor.get_dlq_messages(count=1)

if dlq_messages:
    dlq_msg = dlq_messages[0]
    dlq_id = dlq_msg.get("id")
    
    print(f"Replaying DLQ message: {dlq_id}")
    
    try:
        new_id = await processor.replay_dlq_message(dlq_id)
        print(f"  Replayed to main stream with new ID: {new_id}")
    except ValueError as e:
        print(f"  Error: {e}")
else:
    print("No DLQ messages to replay")

## 8. Simulating Crash Recovery

Demonstrate how the processor recovers from a simulated crash.

In [ ]:
# Add more messages
client = Redis.from_url(REDIS_URL, decode_responses=True)
for i in range(3):
    await client.xadd(STREAM_NAME, {
        "type": "crash_test",
        "message_num": str(i),
    })
await client.aclose()

# Create a "crashed" consumer that reads but doesn't ACK
crashed_processor = RobustStreamProcessor(
    redis_url=REDIS_URL,
    stream_name=STREAM_NAME,
    consumer_group="demo_workers",
    consumer_name="crashed_worker",
    claim_timeout_ms=3000,  # Short timeout for demo
)
await crashed_processor.initialize()

# Read messages but don't process (simulating crash)
print("Simulating crashed worker reading messages...")
client = Redis.from_url(REDIS_URL, decode_responses=True)
msgs = await client.xreadgroup(
    "demo_workers",
    "crashed_worker",
    {STREAM_NAME: ">"},
    count=3,
)
await client.aclose()

if msgs:
    print(f"Crashed worker claimed {len(msgs[0][1])} messages (not ACKed)")
else:
    print("No new messages to read")

await crashed_processor.close()

In [ ]:
# Wait for claim timeout
print("Waiting for claim timeout (3 seconds)...")
await asyncio.sleep(4)

# Recovery worker claims the abandoned messages
print("\nRecovery worker claiming abandoned messages...")
claimed = await processor.claim_pending_messages()
print(f"Claimed {claimed} messages from crashed worker")

# Check health
stats = await processor.get_health_stats()
print(f"\nPending after recovery: {stats['pending_messages']}")

## Summary

The `RobustStreamProcessor` provides:

1. **Batch Processing** - Process messages with custom handlers
2. **Crash Recovery** - Claim pending messages from failed consumers
3. **Dead-Letter Queue** - Store failed messages for later analysis
4. **Message Replay** - Retry failed messages after fixing issues
5. **Health Monitoring** - Track stream and processor health

Use this for:
- Production-grade message processing
- Fault-tolerant distributed systems
- Handling transient failures with automatic retry

In [ ]:
# Cleanup
await processor.close()
print("Processor closed.")